In [0]:
# CELL 1 — IMPORTS
import json
import os
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
from pyspark.sql.functions import col, current_date, when, explode_outer, lit
from delta.tables import DeltaTable
# CELL 6 — silver_yf_prices (re-fetch from yfinance directly)
# Cleaner than trying to unpivot the wide JSON format
print("--- silver_yf_prices ---")

%pip install yfinance --quiet
dbutils.library.restartPython()
print("Libraries loaded")

In [0]:
# VERIFICATION — run in Silver notebook
CH_PATH = "dbfs:/Volumes/corporate_data_lakehouse/bronze/raw_comp_data/company_house"
YF_PATH = "dbfs:/Volumes/corporate_data_lakehouse/bronze/raw_comp_data/yfinance"

print("=" * 45)
print("  BRONZE LAYER — FILE COUNT CHECK")
print("=" * 45)

for label, path in [
    ("CH overview",       f"{CH_PATH}/overview"),
    ("CH filing_history", f"{CH_PATH}/filing_history"),
    ("CH people",         f"{CH_PATH}/people"),
    ("YF history",        f"{YF_PATH}/history"),
    ("YF stats",          f"{YF_PATH}/stats"),
    ("YF income",         f"{YF_PATH}/income_statement"),
    ("YF balance",        f"{YF_PATH}/balance_sheet"),
    ("YF cashflow",       f"{YF_PATH}/cashflow"),
]:
    try:
        files = [f for f in dbutils.fs.ls(path) if f.name.endswith(".json")]
        status = "OK" if len(files) == 20 else "CHECK"
        print(f"  {label:<22} — {len(files):>2} files  {status}")
    except Exception as e:
        print(f"  {label:<22} — MISSING")

print("=" * 45)
print("  All 8 folders should show 20 files OK")
print("=" * 45)

In [0]:
# CELL 4 — silver_ch_officers
print("--- silver_ch_officers ---")

raw = (spark.read
           .option("multiLine", "true")
           .option("mergeSchema", "true")
           .json(f"{CH_PEOPLE}/*.json"))

silver_officers = (raw
    .select(F.explode("items").alias("officer"))
    .select(
        F.regexp_extract(
            F.col("_metadata.file_path"), r"(\d{8})\.json$", 1
        ).alias("company_number"),
        F.col("officer.name").alias("officer_name"),
        F.upper(F.trim(F.col("officer.name"))).alias("officer_name_clean"),
        F.col("officer.officer_role").alias("role"),
        F.col("officer.date_of_birth.month").cast("int").alias("dob_month"),
        F.col("officer.date_of_birth.year").cast("int").alias("dob_year"),
        F.col("officer.appointed_on").cast("date").alias("appointed_date"),
        F.col("officer.resigned_on").cast("date").alias("resigned_date"),
        F.when(
            F.col("officer.resigned_on").isNull(), "active"
        ).otherwise("resigned").alias("officer_status"),
        F.col("officer.nationality").alias("nationality"),
        F.col("officer.country_of_residence").alias("country_of_residence"),
        F.current_timestamp().alias("silver_ts")
    )
    .dropDuplicates(["company_number", "officer_name_clean", "role", "appointed_date"])
)

(silver_officers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DB}.silver_ch_officers"))

print(f"Rows written: {silver_officers.count()}")
silver_officers.show(5, truncate=False)

In [0]:
# CELL 5 — silver_ch_filings
print("--- silver_ch_filings ---")

raw = (spark.read
           .option("multiLine", "true")
           .option("mergeSchema", "true")
           .json(f"{CH_FILINGS}/*.json"))

silver_filings = (raw
    .select(F.explode("items").alias("filing"))
    .select(
        F.regexp_extract(
            F.col("_metadata.file_path"), r"(\d{8})\.json$", 1
        ).alias("company_number"),
        F.col("filing.category").alias("category"),
        F.col("filing.type").alias("filing_type"),
        F.col("filing.description").alias("description"),
        F.col("filing.date").cast("date").alias("filed_date"),
        F.col("filing.action_date").cast("date").alias("action_date"),
        F.current_timestamp().alias("silver_ts")
    )
    .dropDuplicates(["company_number", "filing_type", "filed_date"])
)

(silver_filings.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DB}.silver_ch_filings"))

print(f"Rows written: {silver_filings.count()}")
silver_filings.show(5, truncate=False)

In [0]:
# Run this after the restart
import yfinance as yf
import pandas as pd
from pyspark.sql import functions as F

print("--- silver_yf_prices ---")

TICKERS = [
    "SHEL.L","HSBA.L","BP.L","ULVR.L","VOD.L",
    "BARC.L","AZN.L","SGE.L","GRG.L","BWY.L",
    "RSW.L","PNN.L","SVT.L","IMI.L","BBY.L",
    "DRX.L","TEP.L","CNA.L","NCC.L","SBRY.L"
]

TICKER_META = {
    "SHEL.L":"Shell plc","HSBA.L":"HSBC Holdings plc",
    "BP.L":"BP p.l.c.","ULVR.L":"Unilever PLC",
    "VOD.L":"Vodafone Group Plc","BARC.L":"Barclays PLC",
    "AZN.L":"AstraZeneca PLC","SGE.L":"The Sage Group plc",
    "GRG.L":"Greggs plc","BWY.L":"Bellway plc",
    "RSW.L":"Renishaw plc","PNN.L":"Pennon Group plc",
    "SVT.L":"Severn Trent plc","IMI.L":"IMI plc",
    "BBY.L":"Balfour Beatty plc","DRX.L":"Drax Group plc",
    "TEP.L":"Telecom Plus plc","CNA.L":"Centrica plc",
    "NCC.L":"NCC Group plc","SBRY.L":"J Sainsbury plc"
}

DB = "fraud_analytics"
all_dfs = []

for ticker in TICKERS:
    try:
        df = yf.Ticker(ticker).history(period="5y").reset_index()
        df.columns = [c.replace(" ", "_").lower() for c in df.columns]

        # Normalise the date — strip timezone info
        df["trade_date"]   = pd.to_datetime(df["date"]).dt.date
        df["ticker"]       = ticker
        df["company_name"] = TICKER_META.get(ticker, "Unknown")

        df = df[["ticker", "company_name", "trade_date",
                 "open", "high", "low", "close",
                 "volume", "dividends", "stock_splits"]]

        all_dfs.append(df)
        print(f"  {ticker}: {len(df)} rows")

    except Exception as e:
        print(f"  {ticker} FAILED: {e}")

# Combine all into one dataframe
combined = pd.concat(all_dfs, ignore_index=True)
print(f"\nTotal rows: {len(combined):,}")

# Convert to Spark and write Delta table
silver_prices = (spark.createDataFrame(combined)
    .withColumn("trade_date",   F.col("trade_date").cast("date"))
    .withColumn("open",         F.col("open").cast("double"))
    .withColumn("high",         F.col("high").cast("double"))
    .withColumn("low",          F.col("low").cast("double"))
    .withColumn("close",        F.col("close").cast("double"))
    .withColumn("volume",       F.col("volume").cast("long"))
    .withColumn("dividends",    F.col("dividends").cast("double"))
    .withColumn("stock_splits", F.col("stock_splits").cast("double"))
    .withColumn("silver_ts",    F.current_timestamp())
    .filter(F.col("close").isNotNull())
    .dropDuplicates(["ticker", "trade_date"])
    .orderBy("ticker", "trade_date")
)

(silver_prices.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("ticker")
    .saveAsTable(f"{DB}.silver_yf_prices"))

print(f"\nRows written: {silver_prices.count():,}")
silver_prices.show(5, truncate=False)

In [0]:
# CELL 7 — silver_yf_income, silver_yf_balance, silver_yf_cashflow
# Same approach as prices — fetch directly from yfinance
print("--- silver_yf_financials ---")

import yfinance as yf
import pandas as pd
from pyspark.sql import functions as F

DB = "fraud_analytics"

TICKERS = [
    "SHEL.L","HSBA.L","BP.L","ULVR.L","VOD.L",
    "BARC.L","AZN.L","SGE.L","GRG.L","BWY.L",
    "RSW.L","PNN.L","SVT.L","IMI.L","BBY.L",
    "DRX.L","TEP.L","CNA.L","NCC.L","SBRY.L"
]

TICKER_META = {
    "SHEL.L":"Shell plc","HSBA.L":"HSBC Holdings plc",
    "BP.L":"BP p.l.c.","ULVR.L":"Unilever PLC",
    "VOD.L":"Vodafone Group Plc","BARC.L":"Barclays PLC",
    "AZN.L":"AstraZeneca PLC","SGE.L":"The Sage Group plc",
    "GRG.L":"Greggs plc","BWY.L":"Bellway plc",
    "RSW.L":"Renishaw plc","PNN.L":"Pennon Group plc",
    "SVT.L":"Severn Trent plc","IMI.L":"IMI plc",
    "BBY.L":"Balfour Beatty plc","DRX.L":"Drax Group plc",
    "TEP.L":"Telecom Plus plc","CNA.L":"Centrica plc",
    "NCC.L":"NCC Group plc","SBRY.L":"J Sainsbury plc"
}

def fetch_financial(fetch_fn, table_name, label):
    print(f"\n  --- {table_name} ---")
    all_dfs = []

    for ticker in TICKERS:
        try:
            df = fetch_fn(yf.Ticker(ticker))

            if df is None or df.empty:
                print(f"    {ticker}: no data")
                continue

            # Transpose — columns become rows (one row per fiscal year)
            df = df.T.reset_index()
            df = df.rename(columns={"index": "fiscal_year"})

            # Clean column names
            df.columns = [
                c.strip().lower()
                 .replace(" ", "_")
                 .replace("/", "_")
                 .replace("-", "_")
                 .replace("(", "")
                 .replace(")", "")
                for c in df.columns
            ]

            df["fiscal_year"]  = pd.to_datetime(df["fiscal_year"]).dt.date
            df["ticker"]       = ticker
            df["company_name"] = TICKER_META.get(ticker, "Unknown")

            all_dfs.append(df)
            print(f"    {ticker}: {len(df)} fiscal years")

        except Exception as e:
            print(f"    {ticker} FAILED: {e}")

    if not all_dfs:
        print(f"  No data for {table_name}")
        return

    combined = pd.concat(all_dfs, ignore_index=True)

    # Cast all numeric columns to float — yfinance mixes int/float
    meta_cols = ["fiscal_year", "ticker", "company_name"]
    for col in combined.columns:
        if col not in meta_cols:
            combined[col] = pd.to_numeric(combined[col], errors="coerce")

    spark_df = (spark.createDataFrame(combined)
        .withColumn("fiscal_year", F.col("fiscal_year").cast("date"))
        .withColumn("silver_ts",   F.current_timestamp())
        .dropDuplicates(["ticker", "fiscal_year"])
        .orderBy("ticker", "fiscal_year")
    )

    (spark_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{DB}.{table_name}"))

    print(f"  {table_name}: {spark_df.count()} rows written")
    spark_df.select(
        "ticker", "company_name", "fiscal_year"
    ).show(3, truncate=False)


# Income statement
fetch_financial(
    lambda s: s.financials,
    "silver_yf_income",
    "income"
)

# Balance sheet
fetch_financial(
    lambda s: s.balance_sheet,
    "silver_yf_balance",
    "balance"
)

# Cash flow
fetch_financial(
    lambda s: s.cashflow,
    "silver_yf_cashflow",
    "cashflow"
)

In [0]:
# CELL 8 — silver_yf_stats
print("--- silver_yf_stats ---")

import yfinance as yf
import pandas as pd
from pyspark.sql import functions as F

DB = "fraud_analytics"

TICKERS = [
    "SHEL.L","HSBA.L","BP.L","ULVR.L","VOD.L",
    "BARC.L","AZN.L","SGE.L","GRG.L","BWY.L",
    "RSW.L","PNN.L","SVT.L","IMI.L","BBY.L",
    "DRX.L","TEP.L","CNA.L","NCC.L","SBRY.L"
]

TICKER_META = {
    "SHEL.L":"Shell plc","HSBA.L":"HSBC Holdings plc",
    "BP.L":"BP p.l.c.","ULVR.L":"Unilever PLC",
    "VOD.L":"Vodafone Group Plc","BARC.L":"Barclays PLC",
    "AZN.L":"AstraZeneca PLC","SGE.L":"The Sage Group plc",
    "GRG.L":"Greggs plc","BWY.L":"Bellway plc",
    "RSW.L":"Renishaw plc","PNN.L":"Pennon Group plc",
    "SVT.L":"Severn Trent plc","IMI.L":"IMI plc",
    "BBY.L":"Balfour Beatty plc","DRX.L":"Drax Group plc",
    "TEP.L":"Telecom Plus plc","CNA.L":"Centrica plc",
    "NCC.L":"NCC Group plc","SBRY.L":"J Sainsbury plc"
}

all_dfs = []

for ticker in TICKERS:
    try:
        info = yf.Ticker(ticker).info
        if not info:
            print(f"  {ticker}: no data")
            continue

        row = {
            "ticker":           ticker,
            "company_name":     TICKER_META.get(ticker, "Unknown"),
            "full_name":        info.get("longName"),
            "sector":           info.get("sector"),
            "industry":         info.get("industry"),
            "country":          info.get("country"),
            "city":             info.get("city"),
            "employees":        info.get("fullTimeEmployees"),
            "market_cap":       info.get("marketCap"),
            "current_price":    info.get("currentPrice"),
            "currency":         info.get("currency"),
            "week_52_high":     info.get("fiftyTwoWeekHigh"),
            "week_52_low":      info.get("fiftyTwoWeekLow"),
            "pe_ratio":         info.get("trailingPE"),
            "price_to_book":    info.get("priceToBook"),
            "debt_to_equity":   info.get("debtToEquity"),
            "dividend_yield":   info.get("dividendYield"),
            "return_on_equity": info.get("returnOnEquity"),
            "return_on_assets": info.get("returnOnAssets"),
            "beta":             info.get("beta"),
        }
        all_dfs.append(row)
        print(f"  {ticker}: OK — {row.get('full_name')}")

    except Exception as e:
        print(f"  {ticker} FAILED: {e}")

combined = pd.DataFrame(all_dfs)

silver_stats = (spark.createDataFrame(combined)
    .withColumn("employees",        F.col("employees").cast("int"))
    .withColumn("market_cap",       F.col("market_cap").cast("long"))
    .withColumn("current_price",    F.col("current_price").cast("double"))
    .withColumn("week_52_high",     F.col("week_52_high").cast("double"))
    .withColumn("week_52_low",      F.col("week_52_low").cast("double"))
    .withColumn("pe_ratio",         F.col("pe_ratio").cast("double"))
    .withColumn("price_to_book",    F.col("price_to_book").cast("double"))
    .withColumn("debt_to_equity",   F.col("debt_to_equity").cast("double"))
    .withColumn("dividend_yield",   F.col("dividend_yield").cast("double"))
    .withColumn("return_on_equity", F.col("return_on_equity").cast("double"))
    .withColumn("return_on_assets", F.col("return_on_assets").cast("double"))
    .withColumn("beta",             F.col("beta").cast("double"))
    .withColumn("silver_ts",        F.current_timestamp())
    .dropDuplicates(["ticker"])
)

(silver_stats.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DB}.silver_yf_stats"))

print(f"\nRows written: {silver_stats.count()}")
silver_stats.select(
    "ticker", "company_name", "sector",
    "market_cap", "current_price", "pe_ratio"
).show(5, truncate=False)

In [0]:
# CELL 9 — FINAL VERIFICATION
print("\n" + "="*55)
print("  SILVER LAYER — FINAL TABLE SUMMARY")
print("="*55)
print(f"  {'Table':<32}  {'Rows':>8}")
print("-"*55)

tables = [
    "silver_ch_overview",
    "silver_ch_officers",
    "silver_ch_filings",
    "silver_yf_prices",
    "silver_yf_income",
    "silver_yf_balance",
    "silver_yf_cashflow",
    "silver_yf_stats",
]

total_rows = 0
for t in tables:
    try:
        count = spark.table(f"fraud_analytics.{t}").count()
        total_rows += count
        print(f"  {t:<32}  {count:>8,}  OK")
    except Exception as e:
        print(f"  {t:<32}  MISSING — {e}")

print("-"*55)
print(f"  {'Total rows across all tables':<32}  {total_rows:>8,}")
print("="*55)
print("  Silver complete. Ready for Gold.")
print("="*55)

In [0]:
# DEBUG — read file using spark, not pandas directly
test_file = f"{YF_HISTORY}/SHEL.L.json"

# Read as raw text first to see the structure
raw_text = spark.read.text(test_file)
raw_text.show(5, truncate=80)

# Then read as JSON and see what columns Spark infers
raw_json = spark.read.option("multiLine", "true").json(test_file)
print("\nColumns Spark sees:")
for col in raw_json.columns:
    print(f"  {col}: {raw_json.schema[col].dataType}")

print(f"\nTotal columns: {len(raw_json.columns)}")
print("\nFirst row sample:")
raw_json.show(1, truncate=50)

In [0]:
print("--- silver_ch_officers ---")

raw = (spark.read
           .option("multiLine", "true")
           .json(f"{CH_PEOPLE}/*.json"))

silver_officers = (raw
    .select(F.explode("items").alias("officer"))
    .select(
        F.regexp_extract(
            F.input_file_name(), r"(\d{8})\.json$", 1
        ).alias("company_number"),
        F.col("officer.name").alias("officer_name"),
        F.upper(F.trim(F.col("officer.name"))).alias("officer_name_clean"),
        F.col("officer.officer_role").alias("role"),
        F.col("officer.date_of_birth.month").cast("int").alias("dob_month"),
        F.col("officer.date_of_birth.year").cast("int").alias("dob_year"),
        F.col("officer.appointed_on").cast("date").alias("appointed_date"),
        F.col("officer.resigned_on").cast("date").alias("resigned_date"),
        F.when(
            F.col("officer.resigned_on").isNull(), "active"
        ).otherwise("resigned").alias("officer_status"),
        F.col("officer.nationality").alias("nationality"),
        F.col("officer.country_of_residence").alias("country_of_residence"),
        F.current_timestamp().alias("silver_ts")
    )
    .dropDuplicates(["company_number", "officer_name_clean", "role", "appointed_date"])
)

(silver_officers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DB}.silver_ch_officers"))

print(f"Rows written: {silver_officers.count()}")
silver_officers.show(5, truncate=False)

In [0]:
# Find where yfinance files actually landed
YF_ROOT = "dbfs:/Volumes/corporate_data_lakehouse/bronze/raw_comp_data/yfinance"

print("Scanning yfinance folder...\n")

def scan(path, depth=0):
    try:
        items = dbutils.fs.ls(path)
        for item in items:
            indent = "  " * depth
            print(f"{indent}{item.name}  ({item.size} bytes)")
            if item.size == 0:  # folder
                scan(item.path, depth + 1)
    except Exception as e:
        print(f"{'  '*depth}ERROR: {e}")

scan(YF_ROOT)

In [0]:
# Simpler diagnostic
dbutils.fs.ls("/Volumes/corporate_data_lakehouse/bronze/raw_comp_data")

In [0]:
# Drill into company_house
print("=== COMPANY HOUSE ===")
for item in dbutils.fs.ls("dbfs:/Volumes/corporate_data_lakehouse/bronze/raw_comp_data/company_house/"):
    print(item.path)
    try:
        for sub in dbutils.fs.ls(item.path):
            print(f"  {sub.path}")
            try:
                for subsub in dbutils.fs.ls(sub.path):
                    print(f"    {subsub.path}")
                    try:
                        for file in dbutils.fs.ls(subsub.path):
                            print(f"      {file.path}  ({file.size} bytes)")
                    except:
                        pass
            except:
                pass
    except:
        pass

print("\n=== YFINANCE ===")
for item in dbutils.fs.ls("dbfs:/Volumes/corporate_data_lakehouse/bronze/raw_comp_data/yfinance/"):
    print(item.path)
    try:
        for sub in dbutils.fs.ls(item.path):
            print(f"  {sub.path}")
            try:
                for subsub in dbutils.fs.ls(sub.path):
                    print(f"    {subsub.path}  ({subsub.size} bytes)")
            except:
                pass
    except:
        pass

In [0]:
# CELL 3 — SILVER: CH OVERVIEW
# Reads all 20 overview JSON files
# Flattens the nested address into individual columns
# Output: one clean row per company

print("--- silver_ch_overview ---")

overview_path = f"{CH_PATH}/overview/{DATE_PART}"

# Read all JSON files in the folder in one shot
# multiLine=True because each file is one big JSON object
raw = (spark.read
           .option("multiLine", "true")
           .json(f"{overview_path}/*.json"))

# Flatten — pull nested fields up to the top level
# registered_office_address is a struct inside the JSON
silver_overview = (raw.select(
    F.col("company_number"),
    F.col("company_name"),
    F.col("company_status"),
    F.col("date_of_creation").cast("date").alias("incorporated_date"),
    F.col("jurisdiction"),
    F.col("type").alias("company_type"),

    # Nested address — flatten each field out
    F.col("registered_office_address.address_line_1").alias("address_line_1"),
    F.col("registered_office_address.address_line_2").alias("address_line_2"),
    F.col("registered_office_address.locality").alias("city"),
    F.col("registered_office_address.postal_code").alias("postcode"),
    F.col("registered_office_address.country").alias("country"),

    # SIC codes tell you what industry the company is in
    F.col("sic_codes").cast("string").alias("sic_codes"),

    # Accounts info
    F.col("accounts.next_due").cast("date").alias("accounts_next_due"),
    F.col("accounts.last_accounts.made_up_to")
     .cast("date").alias("accounts_last_made_up"),

    # Metadata
    F.lit(DATE_PART).alias("bronze_partition"),
    F.current_timestamp().alias("silver_ts")
))

# Drop duplicates — in case ingestion ran twice
silver_overview = silver_overview.dropDuplicates(["company_number"])

# Write to Delta table
(silver_overview.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DB}.silver_ch_overview"))

count = silver_overview.count()
print(f"silver_ch_overview: {count} rows written")
silver_overview.show(3, truncate=False)

In [0]:


# ========== FUNCTION 1: FLATTEN ALL DATA ==========
def flatten_all_data(df):
    df_exploded = df.withColumn(
        "previous_company_names_exploded",
        explode_outer("previous_company_names")
    )

    return df_exploded.select(
        "company_name",
        "company_number",
        "company_status",
        col("accounts.accounting_reference_date.day").alias("acc_ref_day"),
        col("accounts.accounting_reference_date.month").alias("acc_ref_month"),
        col("accounts.last_accounts.made_up_to").alias("last_made_up_to"),
        col("accounts.last_accounts.period_end_on").alias("last_period_end"),
        col("accounts.last_accounts.period_start_on").alias("last_period_start"),
        col("accounts.last_accounts.type").alias("last_accounts_type"),
        col("accounts.next_accounts.due_on").alias("next_due_on"),
        col("accounts.next_accounts.overdue").alias("next_overdue"),
        col("accounts.next_accounts.period_end_on").alias("next_period_end"),
        col("accounts.next_accounts.period_start_on").alias("next_period_start"),
        col("accounts.next_due").alias("next_due"),
        col("accounts.next_made_up_to").alias("next_made_up_to"),
        col("accounts.overdue").alias("accounts_overdue"),
        col("previous_company_names_exploded.name").alias("previous_company_name"),
        col("previous_company_names_exploded.effective_from").alias("effective_from"),
        col("previous_company_names_exploded.ceased_on").alias("ceased_on"),
        when(col("previous_company_names_exploded.name").isNotNull(), "previous_names")
        .otherwise("accounts").alias("record_type")
    )

# ========== FUNCTION 2: SCD2 MERGE ==========
def scd2_merge(spark, source_df, target_table, business_key):

    source_df = source_df.withColumn("effective_start_date", current_date()) \
                         .withColumn("effective_end_date", lit(None).cast("date")) \
                         .withColumn("is_current", lit(1))

    if not spark.catalog.tableExists(target_table):
        print(f"Creating table: {target_table}")

        source_df.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(target_table)

        print("✓ Table created")

    else:
        print(f"Merging into table: {target_table}")

        delta_table = DeltaTable.forName(spark, target_table)

        # Build null-safe merge condition using <=> operator
        merge_parts = [f"(t.{k} <=> s.{k})" for k in business_key]
        merge_condition = " AND ".join(merge_parts)

        delta_table.alias("t").merge(
            source_df.alias("s"),
            merge_condition + " AND t.is_current = 1"
        ).whenMatchedUpdate(
            set={
                "is_current": "0",
                "effective_end_date": "current_date()"
            }
        ).execute()

        source_df.write.format("delta") \
            .mode("append") \
            .saveAsTable(target_table)

        print("✓ Merge completed")

# ========== MAIN EXECUTION ==========

print("\n[STEP 1] Reading Bronze table...")
df = spark.read.table("corporate_data_lakehouse.bronze.comp_house_overview")
print(f" Loaded {df.count()} records")

print("\n[STEP 2] Flattening data (accounts + previous names)...")
df_final = flatten_all_data(df)

# Remove duplicates after flattening
df_final = df_final.dropDuplicates([
    "company_number",
    "record_type",
    "previous_company_name",
    "effective_from",
    "ceased_on"
])

print(f"Flattened {df_final.count()} records after deduplication")

print("\n[STEP 3] Merging into Silver layer...")
scd2_merge(
    spark,
    df_final,
    "corporate_data_lakehouse.silver.comp_overview",
    ["company_number", "record_type", "previous_company_name", "effective_from", "ceased_on"]
)

print("\n[STEP 4] Verifying results...")
result = spark.table("corporate_data_lakehouse.silver.comp_overview")

print(f"Total records: {result.count()}")
print(f"Total columns: {len(result.columns)}")

print("SCD2 TABLE CREATED SUCCESSFULLY!")